# 생성 실험 순서의 판단 재료

PLAN "생성 실험 순서"(H10·H18·H19·H20·H22·H23·H24, H25)를 정할 재료다. 생성 베이스라인(S25, gen-full 330질의, judge j_v8 `deepseek_go_mixed`)의 실패 유형별 규모, 가설마다 겨냥하는 유형과 질의 수, 겨냥 질의가 모두 고쳐질 때의 종합 지표 최대 개선 폭(최소 개선 폭 δ 0.02 대비, D-53), 가설별 재실행 범위와 시간·비용을 표로 낸다. 새 API 호출 없이 기존 생성 기록·judge 결과만 읽는다. 질의 원문은 출력하지 않는다. 순서는 적지 않는다.

In [1]:
import json
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

from rag.gen_eval.checks import CITE, unlisted_refusal
from rag.gen_eval.common import read_jsonl
from rag.index import load_cfg

In [14]:
ROOT = Path.cwd().parent
STEM = "hybrid-cond-filter-law-ms256-k20-ce-v2-dev-full"
JUDGE_PATH = ROOT / f"data/gen_eval/judge/{STEM}-deepseek_go_mixed-j_v8-n330.jsonl"  # S25 베이스라인(docs/slices/25-blind-baseline.md)
RUNS_PATH = ROOT / f"data/gen_eval/runs/{STEM}.jsonl"  # 생성 기록. 앞 330 + 블라인드 풀 100
CORPUS_PATH = ROOT / "data/processed/corpus_context.jsonl"
DELTA = 0.02  # 최소 개선 폭(D-53 결정 4)
# 질의자 주장을 사실처럼 쓴 진술(H22, verdicts 규칙 5)을 충실성 0 판정 이유에서 찾는 패턴. 질의자를 가리키는 말과 확인·주장을 가리키는 말이 함께 든 이유다. 낱말 대조라 어림이다
CLAIM = re.compile(r"(질의자|위원)")
CLAIM_KIND = re.compile(r"(확인|사실처럼|단정|주장)")
# 실행 시간 측정값. S25: 새 230질의 judge 15분 22초(동시 8, D-51). S26: 거부 세트 생성 319질의 + judge가 13:32~13:55(docs/slices/26-oracle.md)
S25_JUDGE_MIN_PER_Q = (15 + 22 / 60) / 230
S26_TOTAL_MIN, S26_Q = 23, 319

In [15]:
def composite(r: dict) -> float | None:
    """질의 종합 지표(충실성·관련성 단순 평균, D-53 결정 2). 하나라도 없으면 None."""
    f = (r.get("faithfulness") or {}).get("score")
    v = (r.get("relevance") or {}).get("score")
    return None if f is None or v is None else (f + v) / 2


def gain(qids: set[str]) -> float:
    """qids가 모두 종합 지표 1이 되고 다른 질의는 그대로일 때의 평균 상승(최대 개선 폭)."""
    return sum(1 - comp[q] for q in qids if q in comp) / len(comp)

## 데이터 로딩

In [16]:
cfg = load_cfg(["index.scope=full", "+exp=exp017"])
rows = read_jsonl(JUDGE_PATH)
qids = [r["qid"] for r in rows]
runs = {r["qid"]: r for r in read_jsonl(RUNS_PATH) if r["qid"] in set(qids)}
phrase = rows[0]["judge_recipe"]["refusal"]
print(f"judge {len(rows)}질의, 생성 기록 {len(runs)}질의, qid 일치 {set(qids) == set(runs)}")
print(f"judge 기록의 생성 레시피 해시 {len({r['judge_recipe']['gen_recipe_sha256'] for r in rows})}종, 거부 문구 '{phrase}'")

judge 330질의, 생성 기록 330질의, qid 일치 True
judge 기록의 생성 레시피 해시 1종, 거부 문구 '제공된 회의록에서 답을 찾지 못했습니다'


## 질의별 종합 지표

In [17]:
comp = {r["qid"]: composite(r) for r in rows if composite(r) is not None}
values = np.array(list(comp.values()))
print(f"종합 지표 계산 질의 {len(comp)}/{len(rows)}, 평균 {values.mean():.3f}")
print(f"δ {DELTA}를 넘으려면 결손(1 − 종합 지표) 합이 {DELTA * len(comp):.1f} 이상 줄어야 한다. 지금 결손 합 {(1 - values).sum():.1f}")
print(pd.Series(values).round(2).value_counts().sort_index().to_string())

종합 지표 계산 질의 314/330, 평균 0.843
δ 0.02를 넘으려면 결손(1 − 종합 지표) 합이 6.3 이상 줄어야 한다. 지금 결손 합 49.4
0.00      1
0.17      1
0.25      5
0.42      1
0.44      1
0.45      1
0.50     35
0.57      1
0.58      4
0.67      9
0.70      2
0.72      1
0.75     55
0.79      2
0.80      2
0.81      1
0.83     13
0.88      7
0.90      6
0.92      1
0.93      2
1.00    163


## 실패 유형별 규모

질의 수는 그 유형이 하나라도 있는 질의다. "최대 개선 폭"은 그 질의가 모두 종합 지표 1이 되고 다른 질의는 그대로일 때의 평균 상승이다(상한). 전체 거부 질의는 종합 지표에 없어 0으로 잡힌다.

In [18]:
def item_qids(outcome: str) -> tuple[set[str], int]:
    hit = Counter()
    for r in rows:
        for i in (r.get("item_eval") or {}).get("items") or []:
            hit[r["qid"]] += i["outcome"] == outcome
    return {q for q, k in hit.items() if k}, sum(hit.values())


def verdict_zero(r: dict) -> list[dict]:
    return [v for v in (r.get("faithfulness") or {}).get("verdicts", []) if v["verdict"] == 0]


def cited_meetings(run: dict) -> set[tuple]:
    """답변이 인용한 근거의 회의(일자·위원회·회차·차수)."""
    by_n = {c["n"]: c for c in run["contexts"]}
    cited = {int(n) for n in CITE.findall(run["answer"])} & set(by_n)
    return {(by_n[n]["date"], by_n[n]["committee_name"], by_n[n]["meeting_number"], by_n[n]["session_number"]) for n in cited}


def year(date: str) -> str:
    return re.match(r"\d{4}", date or "")[0] if re.match(r"\d{4}", date or "") else date

In [19]:
parts = list(cfg.gen_eval.unlisted_refusal)
types = {}
types["전체 거부(코드)"] = ({r["qid"] for r in rows if r["refusal"] == "full"}, None)
types["부분 거부(코드)"] = ({r["qid"] for r in rows if r["refusal"] == "partial"}, None)
types["지정 문구 없는 거부"] = ({q for q in qids if unlisted_refusal(runs[q]["answer"], phrase, parts)}, None)
for name in ["과잉 거부", "상황 미전달", "근거 없이 답함", "안내 없이 부분 답함", "무언 생략", "누락"]:
    types[f"항목: {name}"] = item_qids(name)
zero = {r["qid"]: verdict_zero(r) for r in rows}
types["충실성 0 진술이 있음"] = ({q for q, v in zero.items() if v}, sum(len(v) for v in zero.values()))
claim = {q: [v for v in vs if CLAIM.search(v["reason"]) and CLAIM_KIND.search(v["reason"])] for q, vs in zero.items()}
types["충실성 0 이유가 질의자 주장(어림)"] = ({q for q, v in claim.items() if v}, sum(len(v) for v in claim.values()))
faith = {r["qid"]: (r.get("faithfulness") or {}).get("score") for r in rows}
types["충실성 < 0.5"] = ({q for q, s in faith.items() if s is not None and s < 0.5}, None)
types["관련성 '일부'"] = ({r["qid"] for r in rows if (r.get("relevance") or {}).get("label") == "일부"}, None)
types["관련성 '답하지 않음'"] = ({r["qid"] for r in rows if (r.get("relevance") or {}).get("label") == "답하지 않음"}, None)
unclear_v = {r["qid"]: sum(v["speaker_unclear"] for v in (r.get("faithfulness") or {}).get("verdicts", [])) for r in rows}
unclear_i = {r["qid"]: sum(i["speaker_unclear"] for i in (r.get("item_eval") or {}).get("items") or []) for r in rows}
types["발언자 구분 불가(진술)"] = ({q for q, k in unclear_v.items() if k}, sum(unclear_v.values()))
types["발언자 구분 불가(항목)"] = ({q for q, k in unclear_i.items() if k}, sum(unclear_i.values()))
meetings = {q: cited_meetings(runs[q]) for q in qids}
types["다른 회의 근거를 함께 인용"] = ({q for q, m in meetings.items() if len(m) > 1}, None)
types["다른 연도 근거를 함께 인용"] = ({q for q, m in meetings.items() if len({year(d) for d, *_ in m}) > 1}, None)
types["정답 청크에 핵심 답 없음"] = ({r["qid"] for r in rows if r["gold_n"] and r["answer_in_chunk"] is False}, None)

fail = pd.DataFrame([{"유형": k, "질의": len(q), "항목·진술": n if n is not None else "", "그중 종합 지표 계산": len(q & set(comp)), "최대 개선 폭": round(gain(q), 3)} for k, (q, n) in types.items()])
fail

,유형,질의,항목·진술,그중 종합 지표 계산,최대 개선 폭
0,전체 거부(코드),9,,0,0.000
1,부분 거부(코드),1,,1,0.000
2,지정 문구 없는 거부,23,,18,0.010
3,항목: 과잉 거부,9,9,7,0.005
4,항목: 상황 미전달,27,28,26,0.010
5,항목: 근거 없이 답함,55,66,55,0.050
6,항목: 안내 없이 부분 답함,50,56,50,0.025
7,항목: 무언 생략,26,31,26,0.022
8,항목: 누락,20,25,18,0.008
9,충실성 0 진술이 있음,107,193,107,0.117


## 가설별 겨냥 유형과 최대 개선 폭

가설마다 겨냥하는 실패 유형의 질의 합집합으로 최대 개선 폭을 어림한다. 상한이라 실제 개선은 이보다 작다. 겨냥 질의가 고쳐져도 다른 질의가 나빠질 수 있고, 생성 잡음(같은 설정 두 실행의 종합 지표 반폭 0.019, D-53)이 더해진다. H24는 종합 지표를 올리는 가설이 아니라 거부 판별을 바로잡는 가설이라 아래에 따로 계산한다.

In [20]:
targets = {
    "H10 컨텍스트 구성": ["항목: 근거 없이 답함", "항목: 안내 없이 부분 답함", "항목: 무언 생략", "항목: 누락", "정답 청크에 핵심 답 없음"],
    "H18 모호·회피 전달": ["항목: 상황 미전달"],
    "H19 거부를 별도 문장으로": ["부분 거부(코드)"],
    "H20 청크에 발언자": ["발언자 구분 불가(진술)", "발언자 구분 불가(항목)"],
    "H22 질의자 주장 구분": ["충실성 0 이유가 질의자 주장(어림)"],
    "H23 회의·연도별로 나눠 답하기": ["다른 연도 근거를 함께 인용"],
    "H24 지정 거부 문구": ["지정 문구 없는 거부"],
}
hyp = []
for h, ts in targets.items():
    q = set().union(*(types[t][0] for t in ts))
    hyp.append({"가설": h, "겨냥 유형": ", ".join(ts), "겨냥 질의": len(q), "최대 개선 폭": round(gain(q), 3), "δ 대비": round(gain(q) / DELTA, 1)})
hyp = pd.DataFrame(hyp)
hyp

,가설,겨냥 유형,겨냥 질의,최대 개선 폭,δ 대비
0,H10 컨텍스트 구성,"항목: 근거 없이 답함, 항목: 안내 없이 부분 답함, 항목: 무언 생략, 항목: ...",163,0.103,5.2
1,H18 모호·회피 전달,항목: 상황 미전달,27,0.010,0.5
2,H19 거부를 별도 문장으로,부분 거부(코드),1,0.000,0.0
3,H20 청크에 발언자,"발언자 구분 불가(진술), 발언자 구분 불가(항목)",89,0.042,2.1
4,H22 질의자 주장 구분,충실성 0 이유가 질의자 주장(어림),94,0.106,5.3
5,H23 회의·연도별로 나눠 답하기,다른 연도 근거를 함께 인용,57,0.027,1.4
6,H24 지정 거부 문구,지정 문구 없는 거부,23,0.010,0.5


H22의 겨냥은 진술 단위라 위 상한(겨냥 질의의 종합 지표가 1이 됨)이 크게 잡힌다. 질의자 주장으로 0을 받은 진술만 1로 바뀌고 관련성과 다른 진술은 그대로일 때의 상승을 따로 계산한다. H20과 겨냥 질의가 얼마나 겹치는지도 본다.

In [26]:
def faith_if_fixed(r: dict) -> float:
    vs = r["faithfulness"]["verdicts"]
    fixed = sum(v["verdict"] == 1 or v in claim[r["qid"]] for v in vs)
    return fixed / len(vs)


by_q = {r["qid"]: r for r in rows}
lift = [(faith_if_fixed(by_q[q]) - faith[q]) / 2 for q in comp if claim[q]]
h20 = set().union(types["발언자 구분 불가(진술)"][0], types["발언자 구분 불가(항목)"][0])
h22 = types["충실성 0 이유가 질의자 주장(어림)"][0]
print(f"질의자 주장 진술만 1로 바뀔 때: 충실성 평균 +{2 * sum(lift) / len(comp):.3f}, 종합 지표 +{sum(lift) / len(comp):.3f} (δ 대비 {sum(lift) / len(comp) / DELTA:.1f})")
print(f"충실성 0 진술 {sum(len(v) for v in zero.values())}개 중 질의자 주장 {sum(len(v) for v in claim.values())}개")
print(f"H22 겨냥 {len(h22)}질의 중 H20 겨냥(발언자 구분 불가)과 겹침 {len(h22 & h20)}질의")

질의자 주장 진술만 1로 바뀔 때: 충실성 평균 +0.172, 종합 지표 +0.086 (δ 대비 4.3)
충실성 0 진술 193개 중 질의자 주장 167개
H22 겨냥 94질의 중 H20 겨냥(발언자 구분 불가)과 겹침 25질의


H19는 내용과 거부 문구를 한 문장에 쓴 경우가 겨냥이다. 코드가 거부로 본 10질의의 거부 문장 중 거부 문구 밖 내용이 있는 문장 수를 센다. H23은 다른 회의를 함께 인용한 것이 곧 맥락을 섞은 것은 아니라(같은 의제를 여러 회의가 다룰 수 있다) 겨냥 질의는 상한이다.

In [21]:
key = re.sub(r"\s+", "", phrase)
extra = [s for r in rows for s in r["refused_sentences"] if len(CITE.sub("", re.sub(r"\s+", "", s)).replace(key, "").strip(".")) > 0]
print(f"거부 문장 {sum(len(r['refused_sentences']) for r in rows)}개 중 거부 문구 밖 내용이 든 문장 {len(extra)}개")
print(f"다른 회의 함께 인용 {len(types['다른 회의 근거를 함께 인용'][0])}질의, 그중 다른 연도 {len(types['다른 연도 근거를 함께 인용'][0])}질의")

거부 문장 10개 중 거부 문구 밖 내용이 든 문장 1개
다른 회의 함께 인용 73질의, 그중 다른 연도 57질의


### H24: 지정 문구 없는 거부가 판정에 주는 영향

지정 문구 없는 거부가 지정 문구로 바뀌면 코드가 거부로 판별해 그 질의가 종합 지표에서 빠진다(짝지은 제외, D-47 결정 6). 그 답변이 지금 받는 점수와 빠질 때의 평균 변화를 본다.

In [22]:
unl = types["지정 문구 없는 거부"][0]
unl_scored = unl & set(comp)
rest = [comp[q] for q in comp if q not in unl]
print(f"지정 문구 없는 거부 {len(unl)}질의 중 종합 지표 계산 {len(unl_scored)}질의, 그 평균 {np.mean([comp[q] for q in unl_scored]):.3f}")
print(f"그 질의를 빼면 종합 지표 평균 {values.mean():.3f} → {np.mean(rest):.3f} ({np.mean(rest) - values.mean():+.3f})")
print(f"관련성 표지: {dict(Counter((r.get('relevance') or {}).get('label') for r in rows if r['qid'] in unl))}")

지정 문구 없는 거부 23질의 중 종합 지표 계산 18질의, 그 평균 0.834
그 질의를 빼면 종합 지표 평균 0.843 → 0.843 (+0.001)
관련성 표지: {'일부': 6, '답함': 13, '답하지 않음': 4}


## 가설별 비용

생성은 OpenRouter Gemma 4 31B(CoreWeave fp4), judge는 opencode Go 구독 한도 안이다(D-38). 생성 비용은 S25 330질의 토큰에 가격을 곱했고, 근거를 바꾸는 가설은 근거 길이 배수로 어림한다. 시간은 S25·S26 측정값을 쓴다. 근거가 길어질 때 judge 지연이 얼마나 늘지는 재지 않았다.

In [23]:
price = cfg.generator.price_per_mtok
tin = sum(r["usage"]["prompt_tokens"] for r in rows)
tout = sum(r["usage"]["completion_tokens"] for r in rows)
gen_cost = (tin * price.input + tout * price.output) / 1e6
print(f"S25 생성 330질의: 입력 {tin:,} · 출력 {tout:,} 토큰, ${gen_cost:.3f}, 질의당 입력 {tin / len(rows):,.0f} 토큰")
print(f"judge 330질의(모든 단계 새로 부름) 어림 {S25_JUDGE_MIN_PER_Q * 330:.0f}분, 생성만 바뀐 실험(답변과 무관한 단계 저장 응답) 생성+judge 어림 {S26_TOTAL_MIN / S26_Q * 330:.0f}분")

S25 생성 330질의: 입력 674,677 · 출력 34,256 토큰, $0.079, 질의당 입력 2,044 토큰
judge 330질의(모든 단계 새로 부름) 어림 22분, 생성만 바뀐 실험(답변과 무관한 단계 저장 응답) 생성+judge 어림 24분


In [24]:
docs = {}
for line in CORPUS_PATH.open(encoding="utf-8"):
    d = json.loads(line)
    docs[d["doc_id"]] = len(d["context"])
chunk_chars = [len(c["text"]) for q in qids for c in runs[q]["contexts"]]
doc_chars = [docs[c["doc_id"]] for q in qids for c in runs[q]["contexts"]]
per_q_doc = [sum(docs[c["doc_id"]] for c in runs[q]["contexts"]) for q in qids]
print(f"근거 청크 글자 수 평균 {np.mean(chunk_chars):,.0f}, 그 문서 전체 평균 {np.mean(doc_chars):,.0f}(중앙값 {np.median(doc_chars):,.0f}), 배수 {np.mean(doc_chars) / np.mean(chunk_chars):.1f}")
print(f"질의당 문서 전체 5개 글자 수 중앙값 {np.median(per_q_doc):,.0f}, p95 {np.percentile(per_q_doc, 95):,.0f}, 최대 {max(per_q_doc):,}")

근거 청크 글자 수 평균 622, 그 문서 전체 평균 4,106(중앙값 2,682), 배수 6.6
질의당 문서 전체 5개 글자 수 중앙값 15,498, p95 55,698, 최대 114,142


In [25]:
ctx_tokens = tin / len(rows)
mult_doc = np.mean(doc_chars) / np.mean(chunk_chars)
cost = pd.DataFrame([
    {"가설": "H10 앞뒤 청크 확장(±1)", "바뀌는 단계": "근거·생성", "judge 재실행": "전 단계(근거가 바뀜)", "생성 비용 어림": f"${gen_cost * 3:.2f}", "시간 어림": f"{S25_JUDGE_MIN_PER_Q * 330:.0f}분 이상(근거 3배, 재지 않음)"},
    {"가설": "H10 문서 전체", "바뀌는 단계": "근거·생성", "judge 재실행": "전 단계", "생성 비용 어림": f"${gen_cost * mult_doc:.2f}", "시간 어림": f"재지 않음(근거 {mult_doc:.0f}배)"},
    {"가설": "H18·H19·H22·H23·H24 프롬프트", "바뀌는 단계": "생성 지시", "judge 재실행": "답변을 읽는 단계(핵심 진술·청크 판정은 저장 응답)", "생성 비용 어림": f"${gen_cost:.2f}", "시간 어림": f"{S26_TOTAL_MIN / S26_Q * 330:.0f}분"},
    {"가설": "H20 청크에 발언자", "바뀌는 단계": "데이터 처리·근거(·judge 입력)", "judge 재실행": "전 단계(근거 문자열이 바뀜)", "생성 비용 어림": f"${gen_cost * 1.1:.2f} 안팎", "시간 어림": "실행은 H10 확장과 비슷, 대조 작업은 미상"},
])
cost

,가설,바뀌는 단계,judge 재실행,생성 비용 어림,시간 어림
0,H10 앞뒤 청크 확장(±1),근거·생성,전 단계(근거가 바뀜),$0.24,"22분 이상(근거 3배, 재지 않음)"
1,H10 문서 전체,근거·생성,전 단계,$0.52,재지 않음(근거 7배)
2,H18·H19·H22·H23·H24 프롬프트,생성 지시,답변을 읽는 단계(핵심 진술·청크 판정은 저장 응답),$0.08,24분
3,H20 청크에 발언자,데이터 처리·근거(·judge 입력),전 단계(근거 문자열이 바뀜),$0.09 안팎,"실행은 H10 확장과 비슷, 대조 작업은 미상"
